# **MODELAGEM DE REDES DINÂMICAS DE COMORBIDADES EM ADULTOS E IDOSOS: UMA ABORDAGEM BASEADA EM GRAFOS E APRENDIZADO DE MÁQUINA UTILIZANDO DADOS LONGITUDINAIS DO HEALTH AND RETIREMENT STUDY (HRS)**

## 1. Leitura e mapeamento de variáveis

In [2]:
!pip -q install pyreadstat pyarrow networkx pillow

import os
import pyreadstat
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

caminho = (
    "/content/drive/MyDrive/Aluno Especial ICMC/"
    "Aprendizado de Máquina para Séries Temporais/"
    "randhrs1992_2022v2.sav"
)

if not os.path.isfile(caminho):
    raise FileNotFoundError(
        f"Dataset não encontrado: {caminho}"
    )

# Leitura apenas dos metadados
_, meta = pyreadstat.read_sav(
    caminho,
    metadataonly=True
)

print("=" * 65)
print("DIAGNÓSTICO RAND HRS")
print("=" * 65)

print(f"Variáveis: {len(meta.column_names):,}")
print(f"Respondentes: {meta.number_rows:,}")
print(f"Tamanho: {os.path.getsize(caminho)/1024**3:.2f} GB")

# Dicionário de variáveis
dicionario = pd.DataFrame({
    "variavel": meta.column_names,
    "descricao": [
        meta.column_names_to_labels.get(v, "")
        for v in meta.column_names
    ]
})

# Busca exploratória
termos = (
    "hypertension|diabetes|cancer|lung|heart|"
    "stroke|psychiatric|arthritis|"
    "high blood pressure"
)

candidatas = dicionario[
    dicionario["descricao"].str.contains(
        termos, case=False, na=False
    )
]

print(f"\nVariáveis candidatas: {len(candidatas)}")
display(candidatas.head(100))

dicionario.to_csv(
    "/content/dicionario_hrs.csv",
    index=False
)

print("\nDiagnóstico concluído.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DIAGNÓSTICO RAND HRS
Variáveis: 21,945
Respondentes: 45,165
Tamanho: 1.09 GB

Variáveis candidatas: 2189


,variavel,descricao
4201,S1DBORLMED,S1DBORLMED:W1 S reports taking oral meds for d...
4202,S1DBINSULN,S1DBINSULN:W1 S reports taking insulin for dia...
4203,S1DBDIAGYR,S1DBDIAGYR:W1 S year diabetes first diagnosed
4205,S1CANCRDR,"S1CANCRDR:W1 Has S seen a dr abt cancer, prv 1..."
4206,S1CANCRYR,S1CANCRYR:W1 S year recent cancer
...,...,...
4340,S5CANRADX,"S5CANRADX:W5 S cancer tx: Radiation/x-ray, prv..."
4341,S5HRTDOC,"S5HRTDOC:W5 S saw Dr. for heart problems, prv ..."
4342,S5HATKRY,S5HATKRY:W5 S most recent Heart Attack Year
4344,S5HATKR,"S5HATKR:W5 S had heart attack, prv 2 yrs/since..."



Diagnóstico concluído.


In [3]:
import re
import pandas as pd

# Reutilizar o dicionário da célula anterior
doencas = {
    "Hipertensão": "HIBPE",
    "Diabetes": "DIABE",
    "Câncer": "CANCRE",
    "Doença pulmonar": "LUNGE",
    "Doença cardíaca": "HEARTE",
    "AVC": "STROKE",
    "Problemas psiquiátricos": "PSYCHE",
    "Artrite": "ARTHRE"
}

resultados = []

for doenca, sufixo in doencas.items():

    # Padrão candidato para variáveis do respondente:
    # R1HIBPE, R2HIBPE, ..., etc.
    padrao = rf"^R(\d+){sufixo}$"

    for _, linha in dicionario.iterrows():
        nome = str(linha["variavel"]).upper()
        match = re.fullmatch(padrao, nome)

        if match:
            resultados.append({
                "doenca": doenca,
                "onda": int(match.group(1)),
                "variavel": nome,
                "descricao": linha["descricao"]
            })

mapeamento = pd.DataFrame(resultados)

if mapeamento.empty:
    print("Nenhuma variável encontrada.")
    print("Será necessário revisar os padrões.")
else:
    mapeamento = mapeamento.sort_values(
        ["doenca", "onda"]
    ).reset_index(drop=True)

    print("=" * 65)
    print("MAPEAMENTO EXPLORATÓRIO DAS DOENÇAS")
    print("=" * 65)

    resumo = (
        mapeamento.groupby("doenca")
        .agg(
            quantidade=("variavel", "count"),
            primeira_onda=("onda", "min"),
            ultima_onda=("onda", "max")
        )
    )

    display(resumo)

    print("\nVariáveis encontradas:")
    display(mapeamento)

    mapeamento.to_csv(
        "/content/mapeamento_candidato_hrs.csv",
        index=False
    )

    print("\nArquivo de mapeamento salvo.")

MAPEAMENTO EXPLORATÓRIO DAS DOENÇAS


,quantidade,primeira_onda,ultima_onda
doenca,,,
AVC,16,1,16
Artrite,16,1,16
Câncer,16,1,16
Diabetes,16,1,16
Doença cardíaca,16,1,16
Doença pulmonar,16,1,16
Hipertensão,16,1,16
Problemas psiquiátricos,16,1,16



Variáveis encontradas:


,doenca,onda,variavel,descricao
0,AVC,1,R1STROKE,R1STROKE:W1 R ever had stroke
1,AVC,2,R2STROKE,R2STROKE:W2 R ever had stroke
2,AVC,3,R3STROKE,R3STROKE:W3 R ever had stroke
3,AVC,4,R4STROKE,R4STROKE:W4 R ever had stroke
4,AVC,5,R5STROKE,R5STROKE:W5 R ever had stroke
...,...,...,...,...
123,Problemas psiquiátricos,12,R12PSYCHE,R12PSYCHE:W12 R ever had psych problems
124,Problemas psiquiátricos,13,R13PSYCHE,R13PSYCHE:W13 R ever had psych problems
125,Problemas psiquiátricos,14,R14PSYCHE,R14PSYCHE:W14 R ever had psych problems
126,Problemas psiquiátricos,15,R15PSYCHE,R15PSYCHE:W15 R ever had psych problems



Arquivo de mapeamento salvo.


In [4]:
import os
import pyreadstat
import pandas as pd

if mapeamento.empty:
    raise ValueError(
        "Nenhuma variável clínica foi identificada."
    )

# Colunas clínicas identificadas no mapeamento
colunas_clinicas = mapeamento["variavel"].unique().tolist()

# Identificadores candidatos do RAND HRS
identificadores_candidatos = [
    "HHIDPN",
    "HHID",
    "PN"
]

identificadores = [
    c for c in identificadores_candidatos
    if c in meta.column_names
]

if not identificadores:
    raise ValueError(
        "Identificador não encontrado. "
        "Verifique o dicionário antes de prosseguir."
    )

# Preservar a ordem original das colunas
selecionadas = set(colunas_clinicas + identificadores)

colunas = [
    c for c in meta.column_names
    if c in selecionadas
]

print(f"Colunas selecionadas: {len(colunas)}")

# Leitura somente das colunas necessárias
df_reduzido, meta_reduzido = pyreadstat.read_sav(
    caminho,
    usecols=colunas,
    apply_value_formats=False
)

print("Dimensões:", df_reduzido.shape)

# Salvando formato reduzido
saida = "/content/hrs_reduzido_preliminar.parquet"

df_reduzido.to_parquet(
    saida,
    engine="pyarrow",
    compression="zstd",
    index=False
)

print("\nREDUÇÃO CONCLUÍDA")
print(
    "Tamanho do arquivo:",
    round(os.path.getsize(saida) / 1024**2, 2),
    "MB"
)


Colunas selecionadas: 131
Dimensões: (45165, 131)

REDUÇÃO CONCLUÍDA
Tamanho do arquivo: 1.17 MB


In [5]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "/content/hrs_reduzido_preliminar.parquet"
)

# Verificar os códigos encontrados em cada doença
auditoria = []

for doenca, grupo in mapeamento.groupby("doenca"):
    colunas_doenca = grupo["variavel"].tolist()

    valores = pd.Series(
        df[colunas_doenca].to_numpy().ravel()
    )

    contagem = valores.value_counts(
        dropna=False
    )

    for codigo, quantidade in contagem.items():
        auditoria.append({
            "doenca": doenca,
            "codigo": str(codigo),
            "quantidade": quantidade
        })

resultado = pd.DataFrame(auditoria)

print("AUDITORIA DOS CÓDIGOS CLÍNICOS")
display(resultado.to_string(index=False))

print("\nIDENTIFICADORES:")
print([
    c for c in df.columns
    if c not in mapeamento["variavel"].values
])

print("\nVALORES AUSENTES POR DOENÇA:")
for doenca, grupo in mapeamento.groupby("doenca"):
    cols = grupo["variavel"].tolist()
    percentual = df[cols].isna().mean().mean() * 100
    print(f"{doenca}: {percentual:.2f}%")


AUDITORIA DOS CÓDIGOS CLÍNICOS


'                 doenca codigo  quantidade\n                    AVC    nan      426510\n                    AVC    0.0      272205\n                    AVC    1.0       23925\n                Artrite    nan      426510\n                Artrite    1.0      155522\n                Artrite    0.0      140608\n                 Câncer    nan      426510\n                 Câncer    0.0      258701\n                 Câncer    1.0       37429\n               Diabetes    nan      426510\n               Diabetes    0.0      236858\n               Diabetes    1.0       59272\n        Doença cardíaca    nan      426510\n        Doença cardíaca    0.0      230597\n        Doença cardíaca    1.0       65533\n        Doença pulmonar    nan      426510\n        Doença pulmonar    0.0      269979\n        Doença pulmonar    1.0       26151\n            Hipertensão    nan      426510\n            Hipertensão    1.0      156617\n            Hipertensão    0.0      139513\nProblemas psiquiátricos    nan 


IDENTIFICADORES:
['HHIDPN', 'HHID', 'PN']

VALORES AUSENTES POR DOENÇA:
AVC: 59.02%
Artrite: 59.02%
Câncer: 59.02%
Diabetes: 59.02%
Doença cardíaca: 59.02%
Doença pulmonar: 59.02%
Hipertensão: 59.02%
Problemas psiquiátricos: 59.02%


## 2. Painel longitudinal e associações Phi

In [6]:
import numpy as np
import pandas as pd
import os

# Validar identificadores
assert df["HHIDPN"].notna().all(), "Identificadores ausentes"
assert df["HHIDPN"].is_unique, "Identificadores duplicados"

# Mapeamento das oito doenças
doencas = sorted(mapeamento["doenca"].unique())
ondas = sorted(mapeamento["onda"].unique())

# Construção eficiente por onda
paineis = []

for onda in ondas:

    grupo = mapeamento[mapeamento["onda"] == onda]

    mapa_colunas = dict(
        zip(grupo["variavel"], grupo["doenca"])
    )

    parte = df[
        ["HHIDPN"] + list(mapa_colunas.keys())
    ].rename(columns=mapa_colunas).copy()

    parte["onda"] = onda

    # Excluir linhas sem nenhuma condição observada
    parte = parte.dropna(
        subset=doencas,
        how="all"
    )

    paineis.append(parte)

painel = pd.concat(
    paineis,
    ignore_index=True
)

# Ordenação longitudinal
painel = painel.sort_values(
    ["HHIDPN", "onda"]
).reset_index(drop=True)

# Tipos compactos, preservando valores ausentes
for doenca in doencas:
    painel[doenca] = painel[doenca].astype("Int8")

painel["onda"] = painel["onda"].astype("int8")

# Validações
assert not painel.duplicated(
    ["HHIDPN", "onda"]
).any()

print("=" * 65)
print("PAINEL LONGITUDINAL CONSTRUÍDO")
print("=" * 65)

print("Dimensões:", painel.shape)
print(
    "Respondentes únicos:",
    painel["HHIDPN"].nunique()
)
print(
    "Ondas:",
    painel["onda"].nunique()
)

print("\nPRIMEIROS REGISTROS")
display(painel.head(15))

# Salvar painel
saida = "/content/hrs_painel_longitudinal.parquet"

painel.to_parquet(
    saida,
    compression="zstd",
    index=False
)

print(
    "\nTamanho do painel:",
    round(os.path.getsize(saida) / 1024**2, 2),
    "MB"
)


PAINEL LONGITUDINAL CONSTRUÍDO
Dimensões: (296130, 10)
Respondentes únicos: 45165
Ondas: 16

PRIMEIROS REGISTROS


,HHIDPN,AVC,Artrite,Câncer,Diabetes,Doença cardíaca,Doença pulmonar,Hipertensão,Problemas psiquiátricos,onda
0,1010.0,0,1,0,1,0,0,1,1,1
1,1010.0,1,1,0,1,1,0,1,1,2
2,2010.0,0,0,0,0,0,1,0,1,1
3,2010.0,0,0,0,0,0,1,0,1,2
4,2010.0,0,1,0,0,0,1,0,1,3
5,2010.0,0,1,0,0,0,1,0,1,4
6,2010.0,0,1,0,0,0,1,0,1,5
7,3010.0,0,0,0,0,1,0,0,0,1
8,3010.0,0,0,0,0,1,0,0,0,2
9,3010.0,0,0,0,0,1,0,0,0,3



Tamanho do painel: 0.53 MB


In [7]:
import numpy as np
import pandas as pd

# Configurações experimentais
TAU = 0.10
MIN_N = 100

doencas = sorted(mapeamento["doenca"].unique())
ondas = sorted(painel["onda"].unique())

matrizes_phi = {}
resultados_arestas = []

for onda in ondas:

    dados_onda = painel.loc[
        painel["onda"] == onda, doencas
    ].astype(float)

    # Correlação de Pearson entre variáveis binárias = Phi
    phi = dados_onda.corr(method="pearson")

    # Ausência de variância produz associação indefinida
    np.fill_diagonal(phi.values, 0.0)

    matrizes_phi[int(onda)] = phi

    n_onda = len(dados_onda)

    for i in range(len(doencas)):
        for j in range(i + 1, len(doencas)):

            valor = phi.iloc[i, j]

            elegivel = (
                n_onda >= MIN_N
                and pd.notna(valor)
            )

            resultados_arestas.append({
                "onda": int(onda),
                "doenca_origem": doencas[i],
                "doenca_destino": doencas[j],
                "n_observado": n_onda,
                "phi": float(valor)
                    if pd.notna(valor) else np.nan,
                "elegivel": elegivel,
                "aresta_ativa": bool(
                    elegivel and valor >= TAU
                )
            })

arestas_phi = pd.DataFrame(resultados_arestas)

resumo_redes = (
    arestas_phi.groupby("onda", as_index=False)
    .agg(
        pares_avaliados=("phi", "count"),
        arestas_ativas=("aresta_ativa", "sum"),
        phi_medio=("phi", "mean")
    )
)

print("RESUMO DAS REDES PHI")
display(resumo_redes.round(4))

print("\nMATRIZ PHI — PRIMEIRA ONDA")
display(matrizes_phi[1].round(3))

print("\nMATRIZ PHI — ÚLTIMA ONDA")
display(matrizes_phi[16].round(3))

arestas_phi.to_csv(
    "/content/arestas_phi_por_onda.csv",
    index=False
)

print("\nTotal de pares avaliados:",
      len(arestas_phi))

print("Matrizes Phi calculadas com sucesso.")


RESUMO DAS REDES PHI


,onda,pares_avaliados,arestas_ativas,phi_medio
0,1,28,12,0.0846
1,2,28,8,0.0750
2,3,28,11,0.0883
3,4,28,11,0.0940
4,5,28,12,0.0954
5,6,28,12,0.0946
6,7,28,12,0.1053
7,8,28,11,0.1018
8,9,28,11,0.1019
9,10,28,14,0.1153



MATRIZ PHI — PRIMEIRA ONDA


,AVC,Artrite,Câncer,Diabetes,Doença cardíaca,Doença pulmonar,Hipertensão,Problemas psiquiátricos
AVC,0.000,0.040,0.019,0.092,0.157,0.057,0.123,0.051
Artrite,0.040,0.000,0.050,0.086,0.115,0.111,0.127,0.121
Câncer,0.019,0.050,0.000,0.007,0.016,0.043,0.008,0.035
Diabetes,0.092,0.086,0.007,0.000,0.140,0.037,0.185,0.056
Doença cardíaca,0.157,0.115,0.016,0.140,0.000,0.152,0.163,0.118
Doença pulmonar,0.057,0.111,0.043,0.037,0.152,0.000,0.042,0.121
Hipertensão,0.123,0.127,0.008,0.185,0.163,0.042,0.000,0.096
Problemas psiquiátricos,0.051,0.121,0.035,0.056,0.118,0.121,0.096,0.000



MATRIZ PHI — ÚLTIMA ONDA


,AVC,Artrite,Câncer,Diabetes,Doença cardíaca,Doença pulmonar,Hipertensão,Problemas psiquiátricos
AVC,0.000,0.096,0.048,0.075,0.194,0.106,0.129,0.089
Artrite,0.096,0.000,0.095,0.094,0.195,0.163,0.183,0.181
Câncer,0.048,0.095,0.000,0.006,0.094,0.063,0.052,0.026
Diabetes,0.075,0.094,0.006,0.000,0.107,0.058,0.229,0.074
Doença cardíaca,0.194,0.195,0.094,0.107,0.000,0.169,0.191,0.107
Doença pulmonar,0.106,0.163,0.063,0.058,0.169,0.000,0.084,0.175
Hipertensão,0.129,0.183,0.052,0.229,0.191,0.084,0.000,0.070
Problemas psiquiátricos,0.089,0.181,0.026,0.074,0.107,0.175,0.070,0.000



Total de pares avaliados: 448
Matrizes Phi calculadas com sucesso.


## 3. MRC-MT e base preditiva

In [8]:
import numpy as np
import pandas as pd

# Parâmetros experimentais iniciais
RHO = 0.80
ALPHA = 0.70
BETA = 0.30
TAU = 0.10

assert np.isclose(ALPHA + BETA, 1.0)

doencas = sorted(mapeamento["doenca"].unique())
ondas = sorted(matrizes_phi.keys())

n = len(doencas)

# Memória temporal inicial
memoria = np.zeros((n, n), dtype=float)

matrizes_mrcmt = {}
matrizes_memoria = {}
historico_mrcmt = []

for onda in ondas:

    phi = matrizes_phi[onda].loc[
        doencas, doencas
    ].to_numpy(dtype=float)

    # Identificar estimativas disponíveis
    elegivel = np.isfinite(phi)

    # Indicador de associação ativa
    ativa = (phi >= TAU).astype(float)

    # Atualizar memória apenas onde Phi é válido
    memoria[elegivel] = (
        RHO * memoria[elegivel]
        + (1 - RHO) * ativa[elegivel]
    )

    # Associação contemporânea positiva
    contemporanea = np.maximum(phi, 0.0)

    # Peso combinado
    W = (
        ALPHA * contemporanea
        + BETA * memoria
    )

    # Preservar ausência de estimativa contemporânea
    W[~elegivel] = np.nan

    # Sem autoarestas
    np.fill_diagonal(memoria, 0.0)
    np.fill_diagonal(W, 0.0)

    matrizes_memoria[onda] = pd.DataFrame(
        memoria.copy(),
        index=doencas,
        columns=doencas
    )

    matrizes_mrcmt[onda] = pd.DataFrame(
        W.copy(),
        index=doencas,
        columns=doencas
    )

    # Registrar pares únicos
    for i in range(n):
        for j in range(i + 1, n):

            historico_mrcmt.append({
                "onda": onda,
                "doenca_1": doencas[i],
                "doenca_2": doencas[j],
                "phi": phi[i, j],
                "memoria": memoria[i, j],
                "peso_mrcmt": W[i, j]
            })

resultado_mrcmt = pd.DataFrame(historico_mrcmt)

print("MRC-MT IMPLEMENTADO")
print("Redes geradas:", len(matrizes_mrcmt))
print("Pares temporais:", len(resultado_mrcmt))

print("\nMATRIZ MRC-MT — ONDA 1")
display(matrizes_mrcmt[1].round(3))

print("\nMATRIZ MRC-MT — ONDA 16")
display(matrizes_mrcmt[16].round(3))

resultado_mrcmt.to_csv(
    "/content/resultados_mrcmt.csv",
    index=False
)

print("\nImplementação concluída.")


MRC-MT IMPLEMENTADO
Redes geradas: 16
Pares temporais: 448

MATRIZ MRC-MT — ONDA 1


,AVC,Artrite,Câncer,Diabetes,Doença cardíaca,Doença pulmonar,Hipertensão,Problemas psiquiátricos
AVC,0.000,0.028,0.013,0.065,0.170,0.040,0.146,0.036
Artrite,0.028,0.000,0.035,0.060,0.141,0.137,0.149,0.145
Câncer,0.013,0.035,0.000,0.005,0.011,0.030,0.006,0.025
Diabetes,0.065,0.060,0.005,0.000,0.158,0.026,0.190,0.039
Doença cardíaca,0.170,0.141,0.011,0.158,0.000,0.166,0.174,0.142
Doença pulmonar,0.040,0.137,0.030,0.026,0.166,0.000,0.029,0.145
Hipertensão,0.146,0.149,0.006,0.190,0.174,0.029,0.000,0.067
Problemas psiquiátricos,0.036,0.145,0.025,0.039,0.142,0.145,0.067,0.000



MATRIZ MRC-MT — ONDA 16


,AVC,Artrite,Câncer,Diabetes,Doença cardíaca,Doença pulmonar,Hipertensão,Problemas psiquiátricos
AVC,0.000,0.219,0.034,0.053,0.428,0.213,0.382,0.184
Artrite,0.219,0.000,0.132,0.112,0.425,0.396,0.420,0.418
Câncer,0.034,0.132,0.000,0.004,0.066,0.044,0.036,0.018
Diabetes,0.053,0.112,0.004,0.000,0.367,0.041,0.452,0.052
Doença cardíaca,0.428,0.425,0.066,0.367,0.000,0.410,0.425,0.176
Doença pulmonar,0.213,0.396,0.044,0.041,0.410,0.000,0.059,0.411
Hipertensão,0.382,0.420,0.036,0.452,0.425,0.059,0.000,0.049
Problemas psiquiátricos,0.184,0.418,0.018,0.052,0.176,0.411,0.049,0.000



Implementação concluída.


In [9]:
import numpy as np
import pandas as pd

doencas = sorted(mapeamento["doenca"].unique())

dados = painel.sort_values(
    ["HHIDPN", "onda"]
).reset_index(drop=True)

grupo = dados.groupby("HHIDPN", sort=False)
proxima_onda = grupo["onda"].shift(-1)

# Somente ondas consecutivas
mascara = proxima_onda.eq(dados["onda"] + 1)

atual = dados.loc[mascara].reset_index(drop=True)

seguinte = pd.DataFrame({
    d: grupo[d].shift(-1).loc[mascara].to_numpy()
    for d in doencas
})

X = atual[doencas].to_numpy(dtype=float)
Y = seguinte[doencas].to_numpy(dtype=float)

registros = []

for onda in sorted(atual["onda"].unique()):

    idx = np.flatnonzero(
        atual["onda"].to_numpy() == onda
    )

    W = matrizes_mrcmt[int(onda)].loc[
        doencas, doencas
    ].to_numpy(dtype=float)

    # Produto matricial: participantes x doenças
    # Valores indefinidos não são tratados como ausência de associação: interromper se houver.
    assert np.isfinite(W).all(), (
        f"Pesos indefinidos na onda {onda}"
    )

    escores = X[idx] @ W

    for j, alvo in enumerate(doencas):

        # Participantes sem relato anterior
        elegiveis = (
            (X[idx, j] == 0)
            & np.isfinite(Y[idx, j])
        )

        posicoes = idx[elegiveis]

        parte = pd.DataFrame({
            "HHIDPN": atual.loc[
                posicoes, "HHIDPN"
            ].to_numpy(),
            "onda": int(onda),
            "doenca_alvo": alvo,
            "escore_mrcmt": escores[
                elegiveis, j
            ],
            "novo_relato": Y[
                posicoes, j
            ].astype(int)
        })

        registros.append(parte)

base_preditiva = pd.concat(
    registros, ignore_index=True
)

print("BASE PREDITIVA EXPLORATÓRIA")
print("Dimensões:", base_preditiva.shape)

print("\nEXEMPLOS")
display(base_preditiva.head(15))

print("\nEVENTOS POR DOENÇA")
display(
    base_preditiva.groupby("doenca_alvo")
    .agg(
        observacoes=("novo_relato", "size"),
        eventos=("novo_relato", "sum"),
        taxa_evento=("novo_relato", "mean"),
        escore_medio=("escore_mrcmt", "mean")
    )
    .round(4)
)

base_preditiva.to_parquet(
    "/content/base_preditiva_exploratoria.parquet",
    index=False
)

print("\nBase exploratória construída.")


BASE PREDITIVA EXPLORATÓRIA
Dimensões: (1502122, 5)

EXEMPLOS


,HHIDPN,onda,doenca_alvo,escore_mrcmt,novo_relato
0,1010.0,1,AVC,0.274365,1
1,2010.0,1,AVC,0.075982,0
2,3010.0,1,AVC,0.170247,0
3,3020.0,1,AVC,0.040894,0
4,10001010.0,1,AVC,0.000000,0
5,10003020.0,1,AVC,0.000000,0
6,10003030.0,1,AVC,0.146150,0
7,10004010.0,1,AVC,0.159369,0
8,10004040.0,1,AVC,0.000000,0
9,10013040.0,1,AVC,0.000000,0



EVENTOS POR DOENÇA


,observacoes,eventos,taxa_evento,escore_medio
doenca_alvo,,,,
AVC,226032,4022,0.0178,0.3229
Artrite,118135,12293,0.1041,0.2506
Câncer,214893,5083,0.0237,0.0841
Diabetes,197979,6136,0.0310,0.2874
Doença cardíaca,193664,7983,0.0412,0.4409
Doença pulmonar,223957,3734,0.0167,0.2732
Hipertensão,119137,9957,0.0836,0.2448
Problemas psiquiátricos,208325,4867,0.0234,0.2577



Base exploratória construída.


In [10]:
import numpy as np
import pandas as pd

doencas = sorted(mapeamento["doenca"].unique())

# Recuperar os dados clínicos da onda de origem
clinicos = painel[
    ["HHIDPN", "onda"] + doencas
].copy()

# Renomear atributos para evitar confusão
mapa_nomes = {
    d: f"clinico_{i}"
    for i, d in enumerate(doencas)
}

clinicos = clinicos.rename(columns=mapa_nomes)

# Associar os estados clínicos aos exemplos preditivos
base_modelagem = base_preditiva.merge(
    clinicos,
    on=["HHIDPN", "onda"],
    how="left",
    validate="many_to_one"
)

atributos_clinicos = list(mapa_nomes.values())

# Verificar integridade
assert not base_modelagem[
    atributos_clinicos
].isna().any().any()

assert len(base_modelagem) == len(base_preditiva)

# A doença-alvo deve estar ausente na origem
mapa_inverso = {
    d: mapa_nomes[d] for d in doencas
}

for doenca in doencas:
    mascara = base_modelagem["doenca_alvo"] == doenca
    assert (
        base_modelagem.loc[
            mascara, mapa_inverso[doenca]
        ] == 0
    ).all()

# Tipos compactos
base_modelagem[atributos_clinicos] = (
    base_modelagem[atributos_clinicos]
    .astype("int8")
)

print("BASE DE MODELAGEM PREPARADA")
print("Dimensões:", base_modelagem.shape)

print("\nMAPEAMENTO DOS ATRIBUTOS")
display(
    pd.DataFrame({
        "doenca": list(mapa_nomes.keys()),
        "atributo": list(mapa_nomes.values())
    })
)

print("\nPRIMEIROS REGISTROS")
display(base_modelagem.head(10))

base_modelagem.to_parquet(
    "/content/base_modelagem_clinica.parquet",
    compression="zstd",
    index=False
)

print("\nPreparação concluída.")


BASE DE MODELAGEM PREPARADA
Dimensões: (1502122, 13)

MAPEAMENTO DOS ATRIBUTOS


,doenca,atributo
0,AVC,clinico_0
1,Artrite,clinico_1
2,Câncer,clinico_2
3,Diabetes,clinico_3
4,Doença cardíaca,clinico_4
5,Doença pulmonar,clinico_5
6,Hipertensão,clinico_6
7,Problemas psiquiátricos,clinico_7



PRIMEIROS REGISTROS


,HHIDPN,onda,doenca_alvo,escore_mrcmt,novo_relato,clinico_0,clinico_1,clinico_2,clinico_3,clinico_4,clinico_5,clinico_6,clinico_7
0,1010.0,1,AVC,0.274365,1,0,1,0,1,0,0,1,1
1,2010.0,1,AVC,0.075982,0,0,0,0,0,0,1,0,1
2,3010.0,1,AVC,0.170247,0,0,0,0,0,1,0,0,0
3,3020.0,1,AVC,0.040894,0,0,1,1,0,0,0,0,0
4,10001010.0,1,AVC,0.000000,0,0,0,0,0,0,0,0,0
5,10003020.0,1,AVC,0.000000,0,0,0,0,0,0,0,0,0
6,10003030.0,1,AVC,0.146150,0,0,0,0,0,0,0,1,0
7,10004010.0,1,AVC,0.159369,0,0,0,1,0,0,0,1,0
8,10004040.0,1,AVC,0.000000,0,0,0,0,0,0,0,0,0
9,10013040.0,1,AVC,0.000000,0,0,0,0,0,0,0,0,0



Preparação concluída.


## 4. Redes rígidas e suavizadas

In [11]:
from scipy.special import expit
# Reutilizo as matrizes Phi já calculadas, sem reconstruir o painel.
phi_historico = {onda: matriz.to_numpy(dtype=float).copy()
                 for onda, matriz in matrizes_phi.items()}
assert len(phi_historico) == 16


In [12]:
import numpy as np
import pandas as pd
from scipy.special import expit

assert "phi_historico" in globals()

RHO = 0.80
ALPHA = 0.70
BETA = 0.30
TAU = 0.10

ondas = sorted(phi_historico.keys())
n = phi_historico[ondas[0]].shape[0]
arestas = np.triu_indices(n, k=1)

configuracoes = {
    "Phi": None,
    "MRCMT_rigido": None,
    "MRCMT_suave_k20": 20,
    "MRCMT_suave_k50": 50,
    "MRCMT_suave_k100": 100
}

redes_por_metodo = {}

for nome, k in configuracoes.items():

    memoria = np.zeros((n, n), dtype=float)
    redes = {}

    for onda in ondas:

        phi = phi_historico[onda]
        contemporanea = np.maximum(phi, 0.0)

        if nome == "Phi":
            W = contemporanea.copy()

        else:
            if nome == "MRCMT_rigido":
                ativacao = (
                    phi >= TAU
                ).astype(float)
            else:
                ativacao = expit(
                    k * (phi - TAU)
                )

            elegivel = np.isfinite(phi)

            memoria[elegivel] = (
                RHO * memoria[elegivel]
                + (1 - RHO) * ativacao[elegivel]
            )

            W = (
                ALPHA * contemporanea
                + BETA * memoria
            )

        W[~np.isfinite(phi)] = np.nan
        np.fill_diagonal(W, 0.0)
        np.fill_diagonal(memoria, 0.0)

        redes[onda] = W.copy()

    redes_por_metodo[nome] = redes

registros = []

for nome, redes in redes_por_metodo.items():

    for anterior, atual in zip(
        ondas[:-1], ondas[1:]
    ):

        a = redes[anterior][arestas]
        b = redes[atual][arestas]

        phi_a = np.maximum(
            phi_historico[anterior], 0.0
        )[arestas]

        phi_b = np.maximum(
            phi_historico[atual], 0.0
        )[arestas]

        mascara = (
            np.isfinite(a)
            & np.isfinite(b)
            & np.isfinite(phi_a)
            & np.isfinite(phi_b)
        )

        a = a[mascara]
        b = b[mascara]

        delta_metodo = b - a
        delta_phi = (
            phi_b[mascara]
            - phi_a[mascara]
        )

        if (
            len(a) >= 2
            and np.std(a) > 0
            and np.std(b) > 0
        ):
            correlacao_redes = np.corrcoef(
                a, b
            )[0, 1]
        else:
            correlacao_redes = np.nan

        if (
            len(delta_metodo) >= 2
            and np.std(delta_metodo) > 0
            and np.std(delta_phi) > 0
        ):
            correlacao_mudancas = np.corrcoef(
                delta_metodo, delta_phi
            )[0, 1]
        else:
            correlacao_mudancas = np.nan

        registros.append({
            "metodo": nome,
            "onda_atual": atual,
            "variacao_media": np.mean(
                np.abs(delta_metodo)
            ),
            "correlacao_redes": correlacao_redes,
            "correlacao_mudancas": correlacao_mudancas,
            "peso_medio": np.mean(b)
        })

resposta_temporal = pd.DataFrame(registros)

resumo_resposta = (
    resposta_temporal
    .groupby("metodo", as_index=False)
    .agg(
        variacao_media=(
            "variacao_media", "mean"
        ),
        correlacao_redes=(
            "correlacao_redes", "mean"
        ),
        correlacao_mudancas=(
            "correlacao_mudancas", "mean"
        ),
        peso_medio=(
            "peso_medio", "mean"
        )
    )
)

print("=" * 65)
print("ESTABILIDADE E RESPOSTA TEMPORAL")
print("=" * 65)

display(resumo_resposta.round(5))

print("\nCORRELAÇÃO DAS MUDANÇAS POR ONDA")
display(
    resposta_temporal.pivot(
        index="onda_atual",
        columns="metodo",
        values="correlacao_mudancas"
    ).round(4)
)

resposta_temporal.to_csv(
    "/content/resposta_temporal.csv",
    index=False
)

resumo_resposta.to_csv(
    "/content/resumo_resposta_temporal.csv",
    index=False
)

print("\nCélula 31 concluída.")


ESTABILIDADE E RESPOSTA TEMPORAL


,metodo,variacao_media,correlacao_redes,correlacao_mudancas,peso_medio
0,MRCMT_rigido,0.01444,0.99009,0.52265,0.17478
1,MRCMT_suave_k100,0.01293,0.99305,0.68296,0.18220
2,MRCMT_suave_k20,0.01175,0.99198,0.87168,0.19097
3,MRCMT_suave_k50,0.01251,0.99344,0.75999,0.18654
4,Phi,0.00809,0.98283,1.00000,0.10408



CORRELAÇÃO DAS MUDANÇAS POR ONDA


metodo,MRCMT_rigido,MRCMT_suave_k100,MRCMT_suave_k20,MRCMT_suave_k50,Phi
onda_atual,,,,,
2,0.6099,0.6228,0.8335,0.6728,1.0
3,0.5787,0.7140,0.8795,0.7613,1.0
4,0.5464,0.6239,0.8075,0.6829,1.0
5,0.3379,0.4420,0.7391,0.5547,1.0
6,0.5437,0.6057,0.7634,0.6511,1.0
7,0.5795,0.7616,0.9303,0.8405,1.0
8,0.4500,0.5666,0.8313,0.6640,1.0
9,0.5309,0.6878,0.8635,0.7495,1.0
10,0.7422,0.8668,0.9834,0.9305,1.0



Célula 31 concluída.


## 5. Robustez estrutural — 100 reamostragens

In [13]:
import numpy as np
import pandas as pd
from scipy.special import expit

DOENCAS = [
    "AVC", "Artrite", "Câncer", "Diabetes",
    "Doença cardíaca", "Doença pulmonar",
    "Hipertensão", "Problemas psiquiátricos"
]

ONDAS_TESTE = [4, 8, 12, 16]
N_BOOT = 100

RHO = 0.80
ALPHA = 0.70
BETA = 0.30
TAU = 0.10

rng = np.random.default_rng(42)

configuracoes = [
    ("Phi", None),
    ("Rigido", None),
    ("Suave", 20),
    ("Suave", 50),
    ("Suave", 100)
]

p = len(DOENCAS)
arestas = np.triu_indices(p, k=1)

def ativacao(phi, metodo, k=None):
    if metodo == "Rigido":
        return (phi >= TAU).astype(float)
    return expit(k * (phi - TAU))

def memoria_historica(onda_limite, metodo, k):
    memoria = np.zeros((p, p), dtype=float)

    for onda in sorted(phi_historico):
        if onda >= onda_limite:
            break

        phi = phi_historico[onda]
        elegivel = np.isfinite(phi)

        a = ativacao(phi, metodo, k)

        memoria[elegivel] = (
            RHO * memoria[elegivel]
            + (1 - RHO) * a[elegivel]
        )

        np.fill_diagonal(memoria, 0.0)

    return memoria

def construir_rede(phi, metodo, k, memoria):
    if metodo == "Phi":
        return np.maximum(phi, 0.0)

    a = ativacao(phi, metodo, k)

    memoria_atual = (
        RHO * memoria
        + (1 - RHO) * a
    )

    W = (
        ALPHA * np.maximum(phi, 0.0)
        + BETA * memoria_atual
    )

    W[~np.isfinite(phi)] = np.nan
    np.fill_diagonal(W, 0.0)

    return W

registros = []

for onda in ONDAS_TESTE:

    print(f"Processando onda {onda}...")

    dados = painel.loc[
        painel["onda"] == onda,
        DOENCAS
    ].to_numpy(dtype=float)

    n = len(dados)

    phi_ref = phi_historico[onda]

    referencias = {}
    memorias = {}

    for metodo, k in configuracoes:

        nome = (
            "Phi" if metodo == "Phi"
            else (
                "MRCMT_rigido"
                if metodo == "Rigido"
                else f"MRCMT_suave_k{k}"
            )
        )

        memoria = (
            None
            if metodo == "Phi"
            else memoria_historica(
                onda, metodo, k
            )
        )

        memorias[nome] = memoria

        referencias[nome] = construir_rede(
            phi_ref, metodo, k, memoria
        )

    for repeticao in range(N_BOOT):

        indices = rng.integers(
            0, n, size=n
        )

        amostra = dados[indices]

        phi_boot = pd.DataFrame(amostra, columns=DOENCAS).corr().to_numpy()

        np.fill_diagonal(phi_boot, 0.0)

        for metodo, k in configuracoes:

            nome = (
                "Phi" if metodo == "Phi"
                else (
                    "MRCMT_rigido"
                    if metodo == "Rigido"
                    else f"MRCMT_suave_k{k}"
                )
            )

            rede_boot = construir_rede(
                phi_boot,
                metodo,
                k,
                memorias[nome]
            )

            a = referencias[nome][arestas]
            b = rede_boot[arestas]

            mascara = (
                np.isfinite(a)
                & np.isfinite(b)
            )

            a = a[mascara]
            b = b[mascara]

            registros.append({
                "onda": onda,
                "repeticao": repeticao,
                "metodo": nome,
                "erro_absoluto": np.mean(
                    np.abs(a-b)
                ),
                "erro_quadratico": np.mean(
                    (a-b)**2
                ),
                "correlacao": (
                    np.corrcoef(a, b)[0, 1]
                    if (
                        len(a) > 1
                        and np.std(a) > 0
                        and np.std(b) > 0
                    )
                    else np.nan
                )
            })

resultados_multionda = pd.DataFrame(registros)

resumo_multionda = (
    resultados_multionda
    .groupby("metodo", as_index=False)
    .agg(
        erro_absoluto_medio=(
            "erro_absoluto", "mean"
        ),
        erro_quadratico_medio=(
            "erro_quadratico", "mean"
        ),
        correlacao_media=(
            "correlacao", "mean"
        )
    )
)

resumo_por_onda = (
    resultados_multionda
    .groupby(["onda", "metodo"], as_index=False)
    .agg(
        erro_absoluto_medio=(
            "erro_absoluto", "mean"
        ),
        correlacao_media=(
            "correlacao", "mean"
        )
    )
)

print("\n" + "=" * 65)
print("ROBUSTEZ EM MÚLTIPLAS ONDAS")
print("=" * 65)

print("\nRESUMO GERAL")
display(resumo_multionda.round(6))

print("\nERRO ABSOLUTO POR ONDA")
display(
    resumo_por_onda.pivot(
        index="onda",
        columns="metodo",
        values="erro_absoluto_medio"
    ).round(6)
)

resultados_multionda.to_csv(
    "/content/robustez_multionda.csv",
    index=False
)

resumo_multionda.to_csv(
    "/content/resumo_robustez_multionda.csv",
    index=False
)

print("\nCélula 30 concluída.")


Processando onda 4...
Processando onda 8...
Processando onda 12...
Processando onda 16...

ROBUSTEZ EM MÚLTIPLAS ONDAS

RESUMO GERAL


,metodo,erro_absoluto_medio,erro_quadratico_medio,correlacao_media
0,MRCMT_rigido,0.007348,0.000254,0.994588
1,MRCMT_suave_k100,0.006748,0.000088,0.997867
2,MRCMT_suave_k20,0.005747,0.000053,0.997012
3,MRCMT_suave_k50,0.006453,0.000070,0.997859
4,Phi,0.006112,0.000059,0.990812



ERRO ABSOLUTO POR ONDA


metodo,MRCMT_rigido,MRCMT_suave_k100,MRCMT_suave_k20,MRCMT_suave_k50,Phi
onda,,,,,
4,0.006334,0.006229,0.005656,0.006232,0.005987
8,0.006317,0.006355,0.005686,0.006360,0.006024
12,0.008553,0.006865,0.005626,0.006362,0.006004
16,0.008189,0.007542,0.006021,0.006857,0.006434



Célula 30 concluída.


## 6. Teste temporal — ondas 2–13 / 14 / 15

In [14]:
import numpy as np
import pandas as pd

assert "base_modelagem" in globals()
assert "redes_por_metodo" in globals()

DOENCAS = [
    "AVC", "Artrite", "Câncer", "Diabetes",
    "Doença cardíaca", "Doença pulmonar",
    "Hipertensão", "Problemas psiquiátricos"
]

ONDA_VALIDACAO = 14
ONDA_TESTE = 15

dados = base_modelagem.copy()

dados = dados[
    dados["onda"].between(2, ONDA_TESTE)
].copy()

def identificar_conjunto(onda):

    if onda < ONDA_VALIDACAO:
        return "Treinamento_interno"

    if onda == ONDA_VALIDACAO:
        return "Validacao"

    if onda == ONDA_TESTE:
        return "Teste_final"

    return "Fora_do_protocolo"

dados["conjunto"] = (
    dados["onda"]
    .apply(identificar_conjunto)
)

auditoria = (
    dados
    .groupby(
        ["doenca_alvo", "conjunto"],
        as_index=False
    )
    .agg(
        observacoes=("novo_relato", "size"),
        eventos=("novo_relato", "sum"),
        participantes=("HHIDPN", "nunique"),
        onda_min=("onda", "min"),
        onda_max=("onda", "max")
    )
)

auditoria["taxa_eventos_pct"] = (
    100
    * auditoria["eventos"]
    / auditoria["observacoes"]
)

# Conferência de separação temporal
assert (
    dados.loc[
        dados["conjunto"] == "Treinamento_interno",
        "onda"
    ].max() <= 13
)

assert (
    dados.loc[
        dados["conjunto"] == "Validacao",
        "onda"
    ].eq(14).all()
)

assert (
    dados.loc[
        dados["conjunto"] == "Teste_final",
        "onda"
    ].eq(15).all()
)

# O escore da onda 15 deve utilizar
# exclusivamente a rede da onda 14.
for metodo in [
    "MRCMT_rigido",
    "MRCMT_suave_k20"
]:
    assert 14 in redes_por_metodo[metodo]

    W = redes_por_metodo[metodo][14]

    assert W.shape == (8, 8)
    assert np.isfinite(W).all()

print("=" * 65)
print("AUDITORIA DO PROTOCOLO PREDITIVO FINAL")
print("=" * 65)

print("\nDISTRIBUIÇÃO POR DOENÇA E CONJUNTO")
display(
    auditoria.sort_values(
        ["doenca_alvo", "onda_min"]
    ).round(4)
)

print("\nRESUMO DOS CONJUNTOS")
resumo_conjuntos = (
    auditoria
    .groupby("conjunto", as_index=False)
    .agg(
        observacoes=("observacoes", "sum"),
        eventos=("eventos", "sum")
    )
)

resumo_conjuntos["taxa_eventos_pct"] = (
    100
    * resumo_conjuntos["eventos"]
    / resumo_conjuntos["observacoes"]
)

display(resumo_conjuntos.round(4))

print("\nPROTOCOLO:")
print("Treinamento interno: ondas 2 a 13")
print("Validação: onda 14")
print("Treinamento final: ondas 2 a 14")
print("Teste reservado: onda 15")
print("Rede utilizada no teste: onda 14")

auditoria.to_csv(
    "/content/auditoria_teste_final.csv",
    index=False
)

resumo_conjuntos.to_csv(
    "/content/resumo_conjuntos_finais.csv",
    index=False
)

print("\nCélula 39 concluída.")


AUDITORIA DO PROTOCOLO PREDITIVO FINAL

DISTRIBUIÇÃO POR DOENÇA E CONJUNTO


,doenca_alvo,conjunto,observacoes,eventos,participantes,onda_min,onda_max,taxa_eventos_pct
1,AVC,Treinamento_interno,190584,3586,35781,2,13,1.8816
2,AVC,Validacao,13130,178,13130,14,14,1.3557
0,AVC,Teste_final,11184,183,11184,15,15,1.6363
4,Artrite,Treinamento_interno,99648,10712,25245,2,13,10.7498
5,Artrite,Validacao,5994,496,5994,14,14,8.2749
3,Artrite,Teste_final,4818,439,4818,15,15,9.1117
7,Câncer,Treinamento_interno,181394,4397,34905,2,13,2.4240
8,Câncer,Validacao,12243,272,12243,14,14,2.2217
6,Câncer,Teste_final,10333,275,10333,15,15,2.6614
10,Diabetes,Treinamento_interno,168741,5179,32581,2,13,3.0692



RESUMO DOS CONJUNTOS


,conjunto,observacoes,eventos,taxa_eventos_pct
0,Teste_final,68813,2602,3.7813
1,Treinamento_interno,1271407,46813,3.6820
2,Validacao,82235,2605,3.1678



PROTOCOLO:
Treinamento interno: ondas 2 a 13
Validação: onda 14
Treinamento final: ondas 2 a 14
Teste reservado: onda 15
Rede utilizada no teste: onda 14

Célula 39 concluída.


In [15]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

assert "base_modelagem" in globals()
assert "redes_por_metodo" in globals()

DOENCAS = [
    "AVC", "Artrite", "Câncer", "Diabetes",
    "Doença cardíaca", "Doença pulmonar",
    "Hipertensão", "Problemas psiquiátricos"
]

ATRIBUTOS = [
    f"clinico_{i}"
    for i in range(len(DOENCAS))
]

CS = [0.001, 0.01, 0.1, 1, 10]

resultados_finais = []
predicoes_finais = []

for indice_alvo, doenca in enumerate(DOENCAS):

    dados = base_modelagem[
        (base_modelagem["doenca_alvo"] == doenca)
        & base_modelagem["onda"].between(2, 15)
    ].copy()

    ondas = dados["onda"].to_numpy(dtype=int)
    y = dados["novo_relato"].to_numpy(dtype=int)

    X_clinico = dados[ATRIBUTOS].to_numpy(
        dtype=float
    )

    configuracoes = {
        "Clinico": X_clinico
    }

    for metodo in [
        "MRCMT_rigido",
        "MRCMT_suave_k20"
    ]:

        escore = np.zeros(len(dados))

        for onda in np.unique(ondas):

            mascara = ondas == onda

            W = redes_por_metodo[
                metodo
            ][int(onda) - 1]

            assert np.isfinite(W).all()

            escore[mascara] = (
                X_clinico[mascara]
                @ W[:, indice_alvo]
            )

        configuracoes[
            f"Clinico_{metodo}"
        ] = np.column_stack([
            X_clinico,
            escore
        ])

    treino_interno = ondas <= 13
    validacao = ondas == 14

    treino_final = ondas <= 14
    teste_final = ondas == 15

    assert treino_interno.sum() > 0
    assert validacao.sum() > 0
    assert teste_final.sum() > 0

    for nome, X in configuracoes.items():

        melhor_C = None
        melhor_roc = -np.inf

        # Seleção de regularização:
        # exclusivamente na onda 14.
        for C in CS:

            modelo = LogisticRegression(
                C=C,
                max_iter=600,
                solver="lbfgs"
            )

            modelo.fit(
                X[treino_interno],
                y[treino_interno]
            )

            prob_val = modelo.predict_proba(
                X[validacao]
            )[:, 1]

            roc_val = roc_auc_score(
                y[validacao],
                prob_val
            )

            if roc_val > melhor_roc:
                melhor_roc = roc_val
                melhor_C = C

        # Reajuste sem utilizar a onda 15.
        modelo_final = LogisticRegression(
            C=melhor_C,
            max_iter=600,
            solver="lbfgs"
        )

        modelo_final.fit(
            X[treino_final],
            y[treino_final]
        )

        prob_teste = modelo_final.predict_proba(
            X[teste_final]
        )[:, 1]

        y_teste = y[teste_final]

        resultados_finais.append({
            "doenca": doenca,
            "modelo": nome,
            "melhor_C": melhor_C,
            "n_teste": int(teste_final.sum()),
            "eventos": int(y_teste.sum()),
            "prevalencia": float(y_teste.mean()),
            "ROC_AUC": roc_auc_score(
                y_teste,
                prob_teste
            ),
            "PR_AUC": average_precision_score(
                y_teste,
                prob_teste
            ),
            "Brier": brier_score_loss(
                y_teste,
                prob_teste
            )
        })

        predicoes_finais.append(
            pd.DataFrame({
                "HHIDPN": dados.loc[
                    teste_final, "HHIDPN"
                ].to_numpy(),
                "doenca": doenca,
                "modelo": nome,
                "y_real": y_teste,
                "probabilidade": prob_teste
            })
        )

resultado_teste_final = pd.DataFrame(
    resultados_finais
)

predicoes_teste_final = pd.concat(
    predicoes_finais,
    ignore_index=True
)

# Comparações pareadas com o modelo clínico.
referencia = resultado_teste_final[
    resultado_teste_final["modelo"] == "Clinico"
][
    ["doenca", "ROC_AUC", "PR_AUC", "Brier"]
].rename(columns={
    "ROC_AUC": "ROC_clinico",
    "PR_AUC": "PR_clinico",
    "Brier": "Brier_clinico"
})

comparacao_final = resultado_teste_final.merge(
    referencia,
    on="doenca",
    how="left",
    validate="many_to_one"
)

comparacao_final["delta_ROC"] = (
    comparacao_final["ROC_AUC"]
    - comparacao_final["ROC_clinico"]
)

comparacao_final["delta_PR"] = (
    comparacao_final["PR_AUC"]
    - comparacao_final["PR_clinico"]
)

comparacao_final["delta_Brier"] = (
    comparacao_final["Brier"]
    - comparacao_final["Brier_clinico"]
)

print("=" * 65)
print("TESTE FINAL RESERVADO — ONDA 15")
print("=" * 65)

print("\nRESULTADOS POR DOENÇA")
display(
    resultado_teste_final.round(5)
)

print("\nDIFERENÇAS EM RELAÇÃO AO CLÍNICO")
display(
    comparacao_final[
        [
            "doenca",
            "modelo",
            "delta_ROC",
            "delta_PR",
            "delta_Brier"
        ]
    ].round(6)
)

print("\nRESUMO DOS DELTAS POR MODELO")
display(
    comparacao_final.groupby(
        "modelo", as_index=False
    )[
        ["delta_ROC", "delta_PR", "delta_Brier"]
    ].mean().round(6)
)

resultado_teste_final.to_csv(
    "/content/resultado_teste_final.csv",
    index=False
)

comparacao_final.to_csv(
    "/content/comparacao_teste_final.csv",
    index=False
)

predicoes_teste_final.to_parquet(
    "/content/predicoes_teste_final.parquet",
    index=False
)

print("\nCélula 40 concluída.")


TESTE FINAL RESERVADO — ONDA 15

RESULTADOS POR DOENÇA


,doenca,modelo,melhor_C,n_teste,eventos,prevalencia,ROC_AUC,PR_AUC,Brier
0,AVC,Clinico,0.001,11184,183,0.01636,0.60801,0.02441,0.01607
1,AVC,Clinico_MRCMT_rigido,0.001,11184,183,0.01636,0.61106,0.02447,0.01607
2,AVC,Clinico_MRCMT_suave_k20,0.010,11184,183,0.01636,0.60762,0.02442,0.01609
3,Artrite,Clinico,0.010,4818,439,0.09112,0.55170,0.10244,0.08334
4,Artrite,Clinico_MRCMT_rigido,0.001,4818,439,0.09112,0.54421,0.10012,0.08306
5,Artrite,Clinico_MRCMT_suave_k20,0.001,4818,439,0.09112,0.54421,0.10011,0.08305
6,Câncer,Clinico,0.100,10333,275,0.02661,0.53621,0.02976,0.02591
7,Câncer,Clinico_MRCMT_rigido,0.100,10333,275,0.02661,0.53614,0.02976,0.02591
8,Câncer,Clinico_MRCMT_suave_k20,0.100,10333,275,0.02661,0.53614,0.02976,0.02591
9,Diabetes,Clinico,0.100,8563,408,0.04765,0.57636,0.06097,0.04540



DIFERENÇAS EM RELAÇÃO AO CLÍNICO


,doenca,modelo,delta_ROC,delta_PR,delta_Brier
0,AVC,Clinico,0.000000,0.000000,0.000000
1,AVC,Clinico_MRCMT_rigido,0.003051,0.000058,0.000005
2,AVC,Clinico_MRCMT_suave_k20,-0.000388,0.000009,0.000024
3,Artrite,Clinico,0.000000,0.000000,0.000000
4,Artrite,Clinico_MRCMT_rigido,-0.007488,-0.002321,-0.000279
5,Artrite,Clinico_MRCMT_suave_k20,-0.007492,-0.002329,-0.000285
6,Câncer,Clinico,0.000000,0.000000,0.000000
7,Câncer,Clinico_MRCMT_rigido,-0.000079,-0.000003,0.000000
8,Câncer,Clinico_MRCMT_suave_k20,-0.000079,-0.000003,0.000000
9,Diabetes,Clinico,0.000000,0.000000,0.000000



RESUMO DOS DELTAS POR MODELO


,modelo,delta_ROC,delta_PR,delta_Brier
0,Clinico,0.000000,0.000000,0.000000
1,Clinico_MRCMT_rigido,-0.000589,-0.000290,-0.000030
2,Clinico_MRCMT_suave_k20,-0.000983,-0.000255,-0.000027



Célula 40 concluída.


## 7. Bootstrap pareado — 300 reamostragens

In [16]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

assert "predicoes_teste_final" in globals()

N_BOOT = 300
SEMENTE = 42

rng = np.random.default_rng(SEMENTE)

MODELO_BASE = "Clinico"

MODELOS_REDE = [
    "Clinico_MRCMT_rigido",
    "Clinico_MRCMT_suave_k20"
]

registros = []

for doenca in sorted(
    predicoes_teste_final["doenca"].unique()
):

    dados = predicoes_teste_final[
        predicoes_teste_final["doenca"] == doenca
    ].copy()

    tabela = dados.pivot(
        index="HHIDPN",
        columns="modelo",
        values="probabilidade"
    )

    desfecho = (
        dados[dados["modelo"] == MODELO_BASE]
        .set_index("HHIDPN")["y_real"]
        .reindex(tabela.index)
    )

    y = desfecho.to_numpy(dtype=int)

    assert len(np.unique(y)) == 2
    assert not tabela.isna().any().any()

    p_base = tabela[
        MODELO_BASE
    ].to_numpy(dtype=float)

    probabilidades_rede = {
        modelo: tabela[modelo].to_numpy(
            dtype=float
        )
        for modelo in MODELOS_REDE
    }

    # Índices comuns aos modelos:
    # bootstrap pareado por participante.
    indices_bootstrap = []

    for _ in range(N_BOOT):

        idx = rng.integers(
            0,
            len(y),
            size=len(y)
        )

        if len(np.unique(y[idx])) == 2:
            indices_bootstrap.append(idx)

    for modelo, p_rede in (
        probabilidades_rede.items()
    ):

        deltas_roc = []
        deltas_pr = []

        for idx in indices_bootstrap:

            y_b = y[idx]

            roc_base = roc_auc_score(
                y_b,
                p_base[idx]
            )

            roc_rede = roc_auc_score(
                y_b,
                p_rede[idx]
            )

            pr_base = average_precision_score(
                y_b,
                p_base[idx]
            )

            pr_rede = average_precision_score(
                y_b,
                p_rede[idx]
            )

            deltas_roc.append(
                roc_rede - roc_base
            )

            deltas_pr.append(
                pr_rede - pr_base
            )

        deltas_roc = np.asarray(
            deltas_roc
        )

        deltas_pr = np.asarray(
            deltas_pr
        )

        delta_roc_observado = (
            roc_auc_score(y, p_rede)
            - roc_auc_score(y, p_base)
        )

        delta_pr_observado = (
            average_precision_score(y, p_rede)
            - average_precision_score(y, p_base)
        )

        registros.append({
            "doenca": doenca,
            "modelo": modelo,
            "n_boot": len(indices_bootstrap),

            "delta_ROC": delta_roc_observado,
            "IC95_ROC_inf": np.percentile(
                deltas_roc, 2.5
            ),
            "IC95_ROC_sup": np.percentile(
                deltas_roc, 97.5
            ),

            "delta_PR": delta_pr_observado,
            "IC95_PR_inf": np.percentile(
                deltas_pr, 2.5
            ),
            "IC95_PR_sup": np.percentile(
                deltas_pr, 97.5
            )
        })

intervalos_finais = pd.DataFrame(
    registros
)

print("=" * 65)
print("BOOTSTRAP PAREADO — TESTE FINAL")
print("=" * 65)

print("\nINTERVALOS DE CONFIANÇA POR DOENÇA")
display(
    intervalos_finais.round(6)
)

print("\nINTERVALOS QUE NÃO INCLUEM ZERO")
intervalos_finais["ROC_exclui_zero"] = (
    (intervalos_finais["IC95_ROC_inf"] > 0)
    | (intervalos_finais["IC95_ROC_sup"] < 0)
)

intervalos_finais["PR_exclui_zero"] = (
    (intervalos_finais["IC95_PR_inf"] > 0)
    | (intervalos_finais["IC95_PR_sup"] < 0)
)

display(
    intervalos_finais[
        [
            "doenca",
            "modelo",
            "ROC_exclui_zero",
            "PR_exclui_zero"
        ]
    ]
)

intervalos_finais.to_csv(
    "/content/intervalos_bootstrap_final.csv",
    index=False
)

print("\nCélula 41 concluída.")


BOOTSTRAP PAREADO — TESTE FINAL

INTERVALOS DE CONFIANÇA POR DOENÇA


,doenca,modelo,n_boot,delta_ROC,IC95_ROC_inf,IC95_ROC_sup,delta_PR,IC95_PR_inf,IC95_PR_sup
0,AVC,Clinico_MRCMT_rigido,300,0.003051,-0.001384,0.008127,0.000058,-0.000557,0.000483
1,AVC,Clinico_MRCMT_suave_k20,300,-0.000388,-0.002352,0.001695,0.000009,-0.000233,0.000398
2,Artrite,Clinico_MRCMT_rigido,300,-0.007488,-0.015464,-0.000375,-0.002321,-0.004494,-0.000212
3,Artrite,Clinico_MRCMT_suave_k20,300,-0.007492,-0.015466,-0.000389,-0.002329,-0.004499,-0.000237
4,Câncer,Clinico_MRCMT_rigido,300,-0.000079,-0.000673,0.000418,-0.000003,-0.000051,0.000036
5,Câncer,Clinico_MRCMT_suave_k20,300,-0.000079,-0.000673,0.000418,-0.000003,-0.000051,0.000036
6,Diabetes,Clinico_MRCMT_rigido,300,-0.001326,-0.003398,0.000612,-0.000038,-0.000968,0.000912
7,Diabetes,Clinico_MRCMT_suave_k20,300,-0.000751,-0.001712,0.000040,0.000034,-0.000377,0.000434
8,Doença cardíaca,Clinico_MRCMT_rigido,300,0.000191,-0.001147,0.001645,0.000071,-0.000299,0.000561
9,Doença cardíaca,Clinico_MRCMT_suave_k20,300,0.000618,-0.000174,0.001387,0.000166,-0.000252,0.000504



INTERVALOS QUE NÃO INCLUEM ZERO


,doenca,modelo,ROC_exclui_zero,PR_exclui_zero
0,AVC,Clinico_MRCMT_rigido,False,False
1,AVC,Clinico_MRCMT_suave_k20,False,False
2,Artrite,Clinico_MRCMT_rigido,True,True
3,Artrite,Clinico_MRCMT_suave_k20,True,True
4,Câncer,Clinico_MRCMT_rigido,False,False
5,Câncer,Clinico_MRCMT_suave_k20,False,False
6,Diabetes,Clinico_MRCMT_rigido,False,False
7,Diabetes,Clinico_MRCMT_suave_k20,False,False
8,Doença cardíaca,Clinico_MRCMT_rigido,False,False
9,Doença cardíaca,Clinico_MRCMT_suave_k20,False,False



Célula 41 concluída.


## 8. Exportação das tabelas

In [17]:
import numpy as np
import pandas as pd
import os

assert "resultado_teste_final" in globals()
assert "comparacao_final" in globals()
assert "intervalos_finais" in globals()

PASTA = "/content/resultados_artigo"
os.makedirs(PASTA, exist_ok=True)

# TABELA 1 — DESEMPENHO FINAL

tabela_1 = resultado_teste_final[
    [
        "doenca",
        "modelo",
        "n_teste",
        "eventos",
        "prevalencia",
        "ROC_AUC",
        "PR_AUC",
        "Brier"
    ]
].copy()

tabela_1 = tabela_1.sort_values(
    ["doenca", "modelo"]
)

# TABELA 2 — COMPARAÇÃO PAREADA

tabela_2 = comparacao_final[
    comparacao_final["modelo"] != "Clinico"
][
    [
        "doenca",
        "modelo",
        "delta_ROC",
        "delta_PR",
        "delta_Brier"
    ]
].merge(
    intervalos_finais[
        [
            "doenca",
            "modelo",
            "IC95_ROC_inf",
            "IC95_ROC_sup",
            "IC95_PR_inf",
            "IC95_PR_sup"
        ]
    ],
    on=["doenca", "modelo"],
    how="left",
    validate="one_to_one"
)

assert not tabela_2.isna().any().any()

# TABELA 3 — SÍNTESE DOS EXPERIMENTOS

sintese = [
    {
        "dimensao": "Predição",
        "indicador": "Delta ROC médio - rígido",
        "valor": comparacao_final.loc[
            comparacao_final["modelo"]
            == "Clinico_MRCMT_rigido",
            "delta_ROC"
        ].mean()
    },
    {
        "dimensao": "Predição",
        "indicador": "Delta ROC médio - suave",
        "valor": comparacao_final.loc[
            comparacao_final["modelo"]
            == "Clinico_MRCMT_suave_k20",
            "delta_ROC"
        ].mean()
    },
    {
        "dimensao": "Predição",
        "indicador": "Delta PR médio - rígido",
        "valor": comparacao_final.loc[
            comparacao_final["modelo"]
            == "Clinico_MRCMT_rigido",
            "delta_PR"
        ].mean()
    },
    {
        "dimensao": "Predição",
        "indicador": "Delta PR médio - suave",
        "valor": comparacao_final.loc[
            comparacao_final["modelo"]
            == "Clinico_MRCMT_suave_k20",
            "delta_PR"
        ].mean()
    }
]

# Robustez: usar apenas resultados previamente
# calculados, sem inventar novos indicadores.

if "resumo_robustez_multionda" in globals():

    robustez = resumo_robustez_multionda.copy()

    print(
        "\nResumo de robustez encontrado "
        "na memória do notebook."
    )

elif os.path.exists(
    "/content/resumo_robustez_multionda.csv"
):

    robustez = pd.read_csv(
        "/content/resumo_robustez_multionda.csv"
    )

    print(
        "\nResumo de robustez carregado "
        "do arquivo."
    )

else:

    robustez = None

    print(
        "\nResumo de robustez não localizado. "
        "Será necessário recuperar a tabela "
        "da Célula 30."
    )

tabela_3 = pd.DataFrame(sintese)

# EXPORTAÇÃO

tabela_1.to_csv(
    f"{PASTA}/tabela_1_desempenho_final.csv",
    index=False
)

tabela_2.to_csv(
    f"{PASTA}/tabela_2_comparacao_bootstrap.csv",
    index=False
)

tabela_3.to_csv(
    f"{PASTA}/tabela_3_sintese_experimentos.csv",
    index=False
)

print("=" * 65)
print("CONSOLIDAÇÃO DOS RESULTADOS DO ARTIGO")
print("=" * 65)

print("\nTABELA 1 — DESEMPENHO FINAL")
display(tabela_1.round(6))

print("\nTABELA 2 — COMPARAÇÃO E IC95%")
display(tabela_2.round(6))

print("\nTABELA 3 — SÍNTESE")
display(tabela_3.round(6))

if robustez is not None:
    print("\nRESULTADOS DE ROBUSTEZ DISPONÍVEIS")
    display(robustez.round(6))

print("\nArquivos exportados para:")
print(PASTA)

print("\nCélula 42 concluída.")



Resumo de robustez carregado do arquivo.
CONSOLIDAÇÃO DOS RESULTADOS DO ARTIGO

TABELA 1 — DESEMPENHO FINAL


,doenca,modelo,n_teste,eventos,prevalencia,ROC_AUC,PR_AUC,Brier
0,AVC,Clinico,11184,183,0.016363,0.608008,0.024408,0.016068
1,AVC,Clinico_MRCMT_rigido,11184,183,0.016363,0.611059,0.024466,0.016073
2,AVC,Clinico_MRCMT_suave_k20,11184,183,0.016363,0.607619,0.024417,0.016092
3,Artrite,Clinico,4818,439,0.091117,0.551701,0.102443,0.083335
4,Artrite,Clinico_MRCMT_rigido,4818,439,0.091117,0.544213,0.100121,0.083057
5,Artrite,Clinico_MRCMT_suave_k20,4818,439,0.091117,0.544209,0.100113,0.083050
6,Câncer,Clinico,10333,275,0.026614,0.536214,0.029760,0.025907
7,Câncer,Clinico_MRCMT_rigido,10333,275,0.026614,0.536135,0.029756,0.025907
8,Câncer,Clinico_MRCMT_suave_k20,10333,275,0.026614,0.536135,0.029756,0.025907
9,Diabetes,Clinico,8563,408,0.047647,0.576363,0.060968,0.045396



TABELA 2 — COMPARAÇÃO E IC95%


,doenca,modelo,delta_ROC,delta_PR,delta_Brier,IC95_ROC_inf,IC95_ROC_sup,IC95_PR_inf,IC95_PR_sup
0,AVC,Clinico_MRCMT_rigido,0.003051,0.000058,0.000005,-0.001384,0.008127,-0.000557,0.000483
1,AVC,Clinico_MRCMT_suave_k20,-0.000388,0.000009,0.000024,-0.002352,0.001695,-0.000233,0.000398
2,Artrite,Clinico_MRCMT_rigido,-0.007488,-0.002321,-0.000279,-0.015464,-0.000375,-0.004494,-0.000212
3,Artrite,Clinico_MRCMT_suave_k20,-0.007492,-0.002329,-0.000285,-0.015466,-0.000389,-0.004499,-0.000237
4,Câncer,Clinico_MRCMT_rigido,-0.000079,-0.000003,0.000000,-0.000673,0.000418,-0.000051,0.000036
5,Câncer,Clinico_MRCMT_suave_k20,-0.000079,-0.000003,0.000000,-0.000673,0.000418,-0.000051,0.000036
6,Diabetes,Clinico_MRCMT_rigido,-0.001326,-0.000038,-0.000032,-0.003398,0.000612,-0.000968,0.000912
7,Diabetes,Clinico_MRCMT_suave_k20,-0.000751,0.000034,-0.000035,-0.001712,0.000040,-0.000377,0.000434
8,Doença cardíaca,Clinico_MRCMT_rigido,0.000191,0.000071,0.000049,-0.001147,0.001645,-0.000299,0.000561
9,Doença cardíaca,Clinico_MRCMT_suave_k20,0.000618,0.000166,-0.000017,-0.000174,0.001387,-0.000252,0.000504



TABELA 3 — SÍNTESE


,dimensao,indicador,valor
0,Predição,Delta ROC médio - rígido,-0.000589
1,Predição,Delta ROC médio - suave,-0.000983
2,Predição,Delta PR médio - rígido,-0.000290
3,Predição,Delta PR médio - suave,-0.000255



RESULTADOS DE ROBUSTEZ DISPONÍVEIS


,metodo,erro_absoluto_medio,erro_quadratico_medio,correlacao_media
0,MRCMT_rigido,0.007348,0.000254,0.994588
1,MRCMT_suave_k100,0.006748,0.000088,0.997867
2,MRCMT_suave_k20,0.005747,0.000053,0.997012
3,MRCMT_suave_k50,0.006453,0.000070,0.997859
4,Phi,0.006112,0.000059,0.990812



Arquivos exportados para:
/content/resultados_artigo

Célula 42 concluída.


## 9. Animação dos grafos dinâmicos

In [18]:
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.animation import FuncAnimation, PillowWriter

saida = Path('/content/resultados_artigo')
saida.mkdir(exist_ok=True)
metodo = 'MRCMT_suave_k20'
redes = redes_por_metodo[metodo]
ondas_animacao = sorted(redes)

# Mantenho os vértices na mesma posição para comparar as ondas.
grafo_base = nx.Graph()
grafo_base.add_nodes_from(doencas)
pos = nx.circular_layout(grafo_base)
fig, ax = plt.subplots(figsize=(9, 7))

def desenhar(onda):
    ax.clear()
    matriz = redes[onda]
    grafo = nx.Graph()
    grafo.add_nodes_from(doencas)
    for i in range(len(doencas)):
        for j in range(i + 1, len(doencas)):
            peso = matriz[i, j]
            if np.isfinite(peso) and peso >= 0.10:
                grafo.add_edge(doencas[i], doencas[j], weight=float(peso))
    nx.draw_networkx_nodes(grafo, pos, ax=ax, node_size=1700, node_color='#cfe8f3')
    nx.draw_networkx_labels(grafo, pos, ax=ax, font_size=8)
    larguras = [2 + 8 * grafo[u][v]['weight'] for u, v in grafo.edges()]
    nx.draw_networkx_edges(grafo, pos, ax=ax, width=larguras, edge_color='#326b9a', alpha=.7)
    ax.set_title(f'Rede de comorbidades — onda {onda} — {metodo}')
    ax.axis('off')

animacao = FuncAnimation(fig, desenhar, frames=ondas_animacao, interval=600, repeat=True)
arquivo_gif = saida / 'redes_dinamicas_hrs.gif'
animacao.save(arquivo_gif, writer=PillowWriter(fps=2))
desenhar(16)
fig.savefig(saida / 'grafo_onda_16.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Arquivos gerados:', arquivo_gif, saida / 'grafo_onda_16.png')


Arquivos gerados: /content/resultados_artigo/redes_dinamicas_hrs.gif /content/resultados_artigo/grafo_onda_16.png


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
